# RideFlow — 01 Landing (marketplace)

Generate FK-consistent synthetic RideFlow data with LakeLogic's simulator and load it into `LANDING.rideflow_*` (all VARCHAR) — the raw drop zone Bronze reads from. In production these would be fed by a managed connector / COPY INTO.


In [ ]:
# LakeLogic installs from Snowflake's internal PyPI mirror (no External Access needed,
# works on trial accounts). Requires >=1.41.0 for the Snowflake-engine fixes
# (set_shared_connection + TRY_CAST->TO_VARCHAR). Deps resolve from the mirror / runtime.
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', 'lakelogic>=1.41.0'], check=True)


In [ ]:
import os, sys
# Resolve the repo root (folder containing domains_rideflow) from the Git-connected
# Workspace checkout, so runtime.* imports and relative contract paths both resolve.
d = os.getcwd()
while d != os.path.dirname(d) and not os.path.isdir(os.path.join(d, 'domains_rideflow')):
    d = os.path.dirname(d)
REPO = d; sys.path.insert(0, REPO); os.chdir(REPO)

DB  = 'RIDEFLOW_DEV_DEMO'
SYS = 'domains_rideflow/marketplace/rideflow/_system.yaml'
os.environ['SNOWFLAKE_DATABASE'] = DB   # contracts resolve {catalog} from this

from snowflake.snowpark.context import get_active_session
session = get_active_session()
conn = session.connection            # reuse THIS session (no password in the notebook)
from lakelogic.engines.snowflake import SnowflakeAdapter
SnowflakeAdapter.set_shared_connection(conn)
print('repo:', REPO)


In [ ]:
from runtime.landing import load_system_landing
loaded = load_system_landing(conn, DB, 'marketplace', 'rideflow', via_stage=True)
import pandas as pd
pd.DataFrame(loaded, columns=['landing_table','rows'])
